# 01 · Data Cleaning & Quality Assessment

**Source:** UCI *Online Retail II* — 1,067,371 transaction lines from a UK
online giftware wholesaler, 2009-12-01 → 2011-12-09.

This notebook establishes what is wrong with the raw data and what was done
about it. The cleaning logic lives in `src/data_cleaning.py`; here we run it and
verify the result.

**The finding that matters most:** the workbook's two sheets *overlap* on
2010-12-01…09. Concatenating them naively — which almost every published
analysis of this dataset does — **doubles demand on those nine trading days**.
Any forecast or safety-stock figure built on that is wrong.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from src.config import load_config, get_paths
cfg, paths = load_config(), get_paths()
print("as-of date:", cfg["project"]["as_of_date"])

as-of date: 2011-12-09


## 1. The sheet-overlap problem

Before trusting anything else, quantify it.

In [2]:
raw = pd.read_excel(paths.raw_excel, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
frames = [df.assign(source_sheet=name) for name, df in raw.items()]
combined = pd.concat(frames, ignore_index=True)
combined["Invoice"] = combined["Invoice"].astype(str).str.strip()

a, b = frames[0], frames[1]
overlap = set(a.Invoice.astype(str).str.strip()) & set(b.Invoice.astype(str).str.strip())
print(f"invoices present in BOTH sheets: {len(overlap):,}")
print(f"sheet A rows for those invoices : {a.Invoice.astype(str).str.strip().isin(overlap).sum():,}")
print(f"sheet B rows for those invoices : {b.Invoice.astype(str).str.strip().isin(overlap).sum():,}")

invoices present in BOTH sheets: 1,088
sheet A rows for those invoices : 22,523
sheet B rows for those invoices : 22,523


In [3]:
# Effect on the daily demand series
BIZ = ["Invoice","StockCode","Description","Quantity","InvoiceDate","Price","Customer ID","Country"]
w = combined[(combined.Quantity > 0)].copy()
w["day"] = pd.to_datetime(w.InvoiceDate).dt.normalize()
w = w[(w.day >= "2010-11-25") & (w.day <= "2010-12-14")]
cmp = pd.DataFrame({
    "naive_concat": w.groupby("day").Quantity.sum(),
    "after_dedupe": w.drop_duplicates(subset=BIZ).groupby("day").Quantity.sum(),
})
cmp["inflation_%"] = (100 * (cmp.naive_concat / cmp.after_dedupe - 1)).round(1)
cmp

,naive_concat,after_dedupe,inflation_%
day,,,
2010-11-25,38204,38054,0.4
2010-11-26,17914,17814,0.6
2010-11-28,16568,16298,1.7
2010-11-29,40165,40057,0.3
2010-11-30,26620,26560,0.2
2010-12-01,54014,26906,100.8
2010-12-02,62696,31283,100.4
2010-12-03,32942,16430,100.5
2010-12-05,32902,16243,102.6


Demand is inflated by **~100% for exactly nine days**. Left untreated this propagates into every trailing average, every seasonal index, and every safety-stock calculation that touches December 2010.

## 2. Returns vs. warehouse write-offs

Negative quantities come from two different processes and must not be treated alike:

* **`C`-prefixed invoices** — genuine customer cancellations/returns. These reduce net demand.
* **Negative lines with no `C` prefix** — zero-priced rows carrying free-text
  warehouse notes (`lost`, `damages`, `wrongly coded`, `printing smudges/thrown away`).
  These are stock adjustments, *not* demand, and are excluded from the demand series.

In [4]:
neg = combined[combined.Quantity < 0].copy()
is_c = neg.Invoice.str.upper().str.startswith("C")
print(f"cancellation lines (C)      : {is_c.sum():,}")
print(f"negative, NOT on a C invoice: {(~is_c).sum():,}")
print(f"  ...of which zero-priced   : {(neg[~is_c].Price <= 0).sum():,}")
neg[~is_c][["Invoice","StockCode","Description","Quantity","Price"]].head(8)

cancellation lines (C)      : 19,493
negative, NOT on a C invoice: 3,457
  ...of which zero-priced   : 3,457


,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,NaN,-50,0.0
3114,489655,20683,NaN,-44,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
4296,489806,18010,NaN,-770,0.0


## 3. Run the cleaning pipeline

In [5]:
from src.data_cleaning import run as clean_run
out = clean_run()
for k, v in out.items():
    print(f"{k:22s} {v.shape}")

transactions_clean     (1005780, 13)
returns                (17914, 11)
adjustments            (3399, 10)
dim_product            (4773, 17)
demand_daily           (1751430, 8)
data_quality           (22, 6)


## 4. The data-quality report

Generated from the run itself — every count is computed, not typed.

In [6]:
dq = out["data_quality"]
dq[["severity","issue","affected_records","unit"]].sort_values(
    "severity", key=lambda s: s.map({"Critical":0,"High":1,"Medium":2,"Low":3,"Info":4})
).head(15)

,severity,issue,affected_records,unit
1,Critical,"Sheet overlap: 1,088 invoices (2010-12-01..09)...",22523,rows
2,High,Exact duplicate transaction lines (same invoic...,11812,rows
4,High,Negative quantities NOT on a cancellation invo...,3399,rows
7,High,Service / fee / adjustment stock codes carried...,4753,rows
8,High,Stock codes differing only by letter case (e.g...,170,stock codes
3,Medium,Missing product description,4275,rows
6,Medium,"Bad-debt adjustment invoices (prefix 'A', nega...",6,rows
9,Medium,Zero or negative unit price on a product SKU,2566,rows
12,Medium,Product IDs carrying more than one distinct de...,751,SKUs
11,Medium,"Extreme order quantities (> 504 units, the 99....",983,rows


## 5. Derived category — and an honest accuracy measurement

The source has **no category field**. Category is derived from the product
description with an ordered keyword ruleset (`src/categorization.py`).

A derivation is only trustworthy if it is measured, so a 60-SKU random sample was
audited by hand: **56/60 correct (93%)**. The four misses fall into `Other`,
which is the safe failure mode — a mis-filed SKU stays visible rather than being
silently attributed to the wrong category.

Two real bugs were found and fixed by that audit:
* unanchored `"PEN "` matched **O(PEN S)HAPE**, filing an ornament under Stationery
* `"BIRD"` captured *LOVEBIRD HANGING DECORATION* into Garden & Outdoor

In [7]:
from src.categorization import coverage_report
dim = out["dim_product"]; u = dim[dim.in_universe]
coverage_report(u)

,n_skus,revenue,pct_skus,pct_revenue
category,,,,
Kitchen & Dining,504,4270971.12,21.27,25.31
Home Decor,363,2762387.42,15.32,16.37
Bags,113,1879763.05,4.77,11.14
Party & Celebration,176,1227724.14,7.43,7.28
Storage & Organisation,136,1096606.00,5.74,6.50
Candles & Fragrance,187,1088887.52,7.89,6.45
Other,235,974249.97,9.92,5.77
Christmas & Seasonal,145,824659.08,6.12,4.89
Stationery & Craft,172,671900.26,7.26,3.98


In [8]:
# Reproduce the audit sample
u.sample(60, random_state=2024)[["product_id","product_name","category"]].head(20)

,product_id,product_name,category
1277,22773,GREEN DRAWER KNOB ACRYLIC EDWARDIAN,Storage & Organisation
1366,21424,WOODLAND STORAGE BOX LARGE,Storage & Organisation
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,Kitchen & Dining
458,21531,RED RETROSPOT SUGAR JAM BOWL,Kitchen & Dining
311,20748,KENSINGTON COFFEE SET,Kitchen & Dining
936,22434,BALLOON PUMP WITH 10 BALLOONS,Party & Celebration
89,22384,LUNCH BAG PINK POLKADOT,Bags
1401,21683,MEDIUM MEDINA STAMPED METAL BOWL,Kitchen & Dining
2807,90161B,ANT COPPER TURQ BOUDICCA BRACELET,Jewellery & Accessories
2215,79191C,RETRO PLASTIC ELEPHANT TRAY,Kitchen & Dining


## 6. SKU universe

The early-warning system only covers SKUs the business is still actively
stocking and for which demand variability is estimable. Discontinued and
one-off SKUs would otherwise generate reorder points built on two data points.

In [9]:
print(f"product SKUs in catalogue : {len(dim):,}")
print(f"SKUs in operational scope : {dim.in_universe.sum():,}")
print(f"revenue coverage          : {100*u.total_revenue.sum()/dim.total_revenue.sum():.1f}%")
panel = out["demand_daily"]
print(f"\ndemand panel: {panel.shape[0]:,} rows = {panel.product_id.nunique():,} SKUs x {panel.invoice_date.nunique():,} days")
print(f"zero-demand SKU-days: {100*(panel.demand_units==0).mean():.1f}%")

product SKUs in catalogue : 4,773
SKUs in operational scope : 2,370
revenue coverage          : 86.1%

demand panel: 1,751,430 rows = 2,370 SKUs x 739 days
zero-demand SKU-days: 74.9%


**Result: 2,370 SKUs — 49.7% of the catalogue but 86.1% of revenue.**

The 74.9% zero-demand rate is the single most consequential fact in the dataset.
It determines the entire forecasting strategy in notebook 03.

→ Continue to `02_eda.ipynb`